Cell 1: load the setup and silence the harmless warnings

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sklearn
print("scikit-learn", sklearn.__version__)

from model_setup import X, y, groups, evaluate

scikit-learn 1.9.0
Rows: 26857 | Schools: 2789
Share labeled high completion: 0.156


Cell 2: baseline and logistic regression

In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

results = {}
results["Dummy"] = evaluate(DummyClassifier(strategy="most_frequent"), "Dummy")
results["Logistic Regression"] = evaluate(
    make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                  LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Logistic Regression")

Dummy {'accuracy': np.float64(0.844), 'precision': np.float64(0.0), 'recall': np.float64(0.0), 'f1': np.float64(0.0), 'roc_auc': np.float64(0.5)}
Logistic Regression {'accuracy': np.float64(0.864), 'precision': np.float64(0.54), 'recall': np.float64(0.883), 'f1': np.float64(0.669), 'roc_auc': np.float64(0.928)}


Cell 3: tree models

In [3]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier

results["Random Forest"] = evaluate(
    make_pipeline(SimpleImputer(strategy="median"),
                  RandomForestClassifier(n_estimators=200, random_state=42,
                                         n_jobs=-1, class_weight="balanced")),
    "Random Forest")

spw = (y == 0).sum() / (y == 1).sum()
results["XGBoost"] = evaluate(
    XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                  scale_pos_weight=spw, random_state=42, eval_metric="logloss"),
    "XGBoost")

try:
    hgb = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
except TypeError:   # older scikit-learn without class_weight support
    hgb = HistGradientBoostingClassifier(random_state=42)
results["HistGradientBoosting"] = evaluate(hgb, "HistGradientBoosting")

Random Forest {'accuracy': np.float64(0.893), 'precision': np.float64(0.645), 'recall': np.float64(0.702), 'f1': np.float64(0.671), 'roc_auc': np.float64(0.945)}
XGBoost {'accuracy': np.float64(0.891), 'precision': np.float64(0.622), 'recall': np.float64(0.769), 'f1': np.float64(0.687), 'roc_auc': np.float64(0.947)}
HistGradientBoosting {'accuracy': np.float64(0.89), 'precision': np.float64(0.617), 'recall': np.float64(0.777), 'f1': np.float64(0.687), 'roc_auc': np.float64(0.944)}


Cell 4: table and save

In [4]:
import pandas as pd

table = pd.DataFrame(results).T
table["label"] = "school completion >= 70% (national)"
display(table)
table.to_csv("results_national.csv")

,accuracy,precision,recall,f1,roc_auc,label
Dummy,0.844,0.000,0.000,0.000,0.500,school completion >= 70% (national)
Logistic Regression,0.864,0.540,0.883,0.669,0.928,school completion >= 70% (national)
Random Forest,0.893,0.645,0.702,0.671,0.945,school completion >= 70% (national)
XGBoost,0.891,0.622,0.769,0.687,0.947,school completion >= 70% (national)
HistGradientBoosting,0.890,0.617,0.777,0.687,0.944,school completion >= 70% (national)
